# Notebook 05 — Azure Databricks Setup & ADLS Integration
**SuperStore Sales & Business Performance Analytics System**

This notebook:
1. Mounts ADLS Gen2 filesystem onto `/mnt/superstore` using **Account Key** auth
2. Verifies all Medallion layers (Bronze / Silver / Gold / Dashboard)
3. Reads Bronze CSV and Silver Parquet into Spark DataFrames
4. Creates and registers **Delta Lake** tables
5. Runs SQL queries and demonstrates **Time Travel**

**Pre-requisite**: Run `python run_pipeline.py` locally first — it uploads all data to ADLS.

```
ADLS Gen2 (sachinxcube / superstore filesystem)
├── bronze/   ← raw CSV
├── silver/   ← cleaned Parquet
├── gold/     ← analytics CSVs
├── dashboard/← PNG charts
└── delta/    ← Delta Lake tables (created here)
```

## Step 0 — Configuration

In [ ]:
# ── SuperStore ADLS Gen2 configuration ───────────────────────────────────
ADLS_ACCOUNT_NAME = "sachinxcube"          # ADLS Gen2 storage account
ADLS_FILESYSTEM   = "superstore"           # ADLS container / filesystem
MOUNT_POINT       = "/mnt/superstore"      # Databricks mount path

# Account key — stored in Databricks Secret Scope
# To create the scope, run from local terminal:
#   databricks secrets create-scope superstore-secrets
#   databricks secrets put-secret superstore-secrets adls-account-key --string-value <key>
ADLS_ACCOUNT_KEY  = dbutils.secrets.get(scope="superstore-secrets", key="adls-account-key")
# ── If secret scope not set up, uncomment and paste key directly:
# ADLS_ACCOUNT_KEY = "<PASTE_YOUR_ADLS_ACCOUNT_KEY_HERE>"

ABFSS_BASE = f"abfss://{ADLS_FILESYSTEM}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net"

# ── Configure Spark for abfss:// (required for Delta table writes) ─────────
spark.conf.set(
    f"fs.azure.account.key.{ADLS_ACCOUNT_NAME}.dfs.core.windows.net",
    ADLS_ACCOUNT_KEY
)

print(f"ADLS account : {ADLS_ACCOUNT_NAME}")
print(f"Filesystem   : {ADLS_FILESYSTEM}")
print(f"Mount point  : {MOUNT_POINT}")
print(f"abfss base   : {ABFSS_BASE}")
print("Spark abfss  : configured")

## Step 1 — Mount ADLS Gen2 onto /mnt/superstore

In [ ]:
# ── Unmount existing mount if present ─────────────────────────────────────
for m in dbutils.fs.mounts():
    if m.mountPoint == MOUNT_POINT:
        dbutils.fs.unmount(MOUNT_POINT)
        print(f"Removed existing mount at {MOUNT_POINT}")
        break

# ── Mount using Account Key auth ───────────────────────────────────────────
dbutils.fs.mount(
    source=f"wasbs://{ADLS_FILESYSTEM}@{ADLS_ACCOUNT_NAME}.blob.core.windows.net/",
    mount_point=MOUNT_POINT,
    extra_configs={
        f"fs.azure.account.key.{ADLS_ACCOUNT_NAME}.blob.core.windows.net": ADLS_ACCOUNT_KEY
    }
)

print(f"\nADLS Gen2 mounted at: {MOUNT_POINT}")
display(dbutils.fs.ls(MOUNT_POINT))

## Step 2 — Verify All Medallion Layers

In [ ]:
layers = ["bronze", "silver", "gold", "dashboard"]
print(f"{'Layer':<14}  {'Files':>6}  {'Total Size':>14}")
print("-" * 40)
for layer in layers:
    path = f"{MOUNT_POINT}/{layer}"
    try:
        files = dbutils.fs.ls(path)
        total_size = sum(f.size for f in files)
        print(f"{layer:<14}  {len(files):>6}  {total_size/1024:>12.1f} KB")
        for f in files:
            print(f"  {f.name}")
    except Exception as e:
        print(f"{layer:<14}  [MISSING] {e}")

## Step 3 — Read Bronze Layer (Raw CSV)

In [ ]:
bronze_path = f"{MOUNT_POINT}/bronze/superstore_sales.csv"

df_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(bronze_path)
)

print(f"Bronze rows  : {df_bronze.count():,}")
print(f"Bronze cols  : {len(df_bronze.columns)}")
df_bronze.printSchema()
display(df_bronze.limit(5))

## Step 4 — Read Silver Layer (Cleaned Parquet)

In [ ]:
silver_path = f"{MOUNT_POINT}/silver/superstore_clean.parquet"

df_silver = spark.read.parquet(silver_path)

print(f"Silver rows  : {df_silver.count():,}")
print(f"Silver cols  : {len(df_silver.columns)}")
df_silver.printSchema()
display(df_silver.limit(5))

## Step 5 — Read Gold Layer (Analytics Aggregations)

In [ ]:
gold_path = f"{MOUNT_POINT}/gold/"

gold_files = dbutils.fs.ls(gold_path)
print(f"{'File':<45}  {'Size (KB)':>10}")
print("-" * 60)
for f in gold_files:
    print(f"{f.name:<45}  {f.size/1024:>10.1f}")

# Read category_performance.csv as sample
df_gold = spark.read.option("header", "true").option("inferSchema", "true") \
    .csv(f"{gold_path}category_performance.csv")
print(f"\ncategory_performance: {df_gold.count()} rows")
display(df_gold)

## Step 6 — Create Delta Lake Tables

In [ ]:
from pyspark.sql import functions as F

delta_base = f"{ABFSS_BASE}/delta/"

# ── Sanitize column names: replace spaces with underscores for Delta compatibility
def sanitize_columns(df):
    return df.toDF(*[c.replace(" ", "_") for c in df.columns])

df_silver_delta = sanitize_columns(df_silver)
print("Sanitized columns:", df_silver_delta.columns[:8], "...")

# ── 1. superstore_clean (Silver → Delta, partitioned) ─────────────────────
print("Creating superstore_clean ...")
df_silver_delta.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("order_year", "Category") \
    .save(delta_base + "superstore_clean/")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS superstore_clean
    USING DELTA LOCATION '{delta_base}superstore_clean/'
""")
print("  Done.")

# ── 2. monthly_trend ──────────────────────────────────────────────────────
print("Creating monthly_trend ...")
df_silver_delta.groupBy(
    F.date_format("Order_Date", "yyyy-MM").alias("year_month")
).agg(
    F.round(F.sum("Sales"),   2).alias("sales"),
    F.round(F.sum("Profit"),  2).alias("profit"),
    F.countDistinct("Order_ID").alias("orders"),
).orderBy("year_month") \
 .write.format("delta").mode("overwrite") \
 .option("overwriteSchema", "true") \
 .save(delta_base + "monthly_trend/")
spark.sql(f"CREATE TABLE IF NOT EXISTS monthly_trend USING DELTA LOCATION '{delta_base}monthly_trend/'")
print("  Done.")

# ── 3. category_performance ───────────────────────────────────────────────
print("Creating category_performance ...")
df_silver_delta.groupBy("Category").agg(
    F.round(F.sum("Sales"),  2).alias("sales"),
    F.round(F.sum("Profit"), 2).alias("profit"),
    F.sum("Quantity").alias("quantity"),
    F.round(F.avg("Discount") * 100, 2).alias("avg_discount_pct"),
    F.countDistinct("Order_ID").alias("orders"),
).write.format("delta").mode("overwrite") \
 .option("overwriteSchema", "true") \
 .save(delta_base + "category_performance/")
spark.sql(f"CREATE TABLE IF NOT EXISTS category_performance USING DELTA LOCATION '{delta_base}category_performance/'")
print("  Done.")

# ── 4. regional_performance ───────────────────────────────────────────────
print("Creating regional_performance ...")
df_silver_delta.groupBy("Region").agg(
    F.round(F.sum("Sales"),  2).alias("sales"),
    F.round(F.sum("Profit"), 2).alias("profit"),
    F.countDistinct("Order_ID").alias("orders"),
    F.countDistinct("Customer_ID").alias("customers"),
).write.format("delta").mode("overwrite") \
 .option("overwriteSchema", "true") \
 .save(delta_base + "regional_performance/")
spark.sql(f"CREATE TABLE IF NOT EXISTS regional_performance USING DELTA LOCATION '{delta_base}regional_performance/'")
print("  Done.")

print("\nAll 4 Delta tables created.")

## Step 7 — Optimize Delta Tables

In [ ]:
for tbl in ["superstore_clean", "monthly_trend", "category_performance", "regional_performance"]:
    print(f"OPTIMIZE {tbl} ...")
    spark.sql(f"OPTIMIZE {tbl}")

spark.sql("SET spark.databricks.delta.retentionDurationCheck.enabled = false")
spark.sql("VACUUM superstore_clean RETAIN 168 HOURS")
print("\nOptimize + Vacuum complete.")
spark.sql("SHOW TABLES").show()

## Step 8 — SQL Analytics on Delta Tables

In [ ]:
print("=== Top 5 Months by Sales ===")
display(spark.sql("""
    SELECT year_month, sales, profit,
           ROUND(profit / sales * 100, 2) AS profit_margin_pct
    FROM   monthly_trend
    ORDER  BY sales DESC
    LIMIT  5
"""))

In [ ]:
print("=== Category Performance ===")
display(spark.sql("""
    SELECT Category, sales, profit,
           ROUND(profit / sales * 100, 2) AS profit_margin_pct,
           avg_discount_pct
    FROM   category_performance
    ORDER  BY sales DESC
"""))

In [ ]:
print("=== Regional Performance ===")
display(spark.sql("SELECT * FROM regional_performance ORDER BY sales DESC"))

In [ ]:
print("=== Top 20 Loss-Making Products ===")
display(spark.sql("""
    SELECT  Product_Name, Category, `Sub-Category`,
            ROUND(Sales,   2) AS sales,
            ROUND(Profit,  2) AS profit,
            Discount, profit_margin
    FROM    superstore_clean
    WHERE   Profit < 0
    ORDER   BY Profit ASC
    LIMIT   20
"""))

## Step 9 — Delta Time Travel

In [ ]:
# Show full write history
display(spark.sql("DESCRIBE HISTORY superstore_clean LIMIT 5"))

In [ ]:
# Time-travel: read version 0 (initial load)
df_v0 = spark.read.format("delta").option("versionAsOf", 0) \
             .load(f"{ABFSS_BASE}/delta/superstore_clean/")
print(f"Version 0 rows: {df_v0.count():,}")

## Summary

| Layer | Path | Format | Purpose |
|-------|------|--------|---------|
| Bronze | `/mnt/superstore/bronze/` | CSV | Raw data |
| Silver | `/mnt/superstore/silver/` | Parquet | Cleaned + typed |
| Gold | `/mnt/superstore/gold/` | CSV | Analytics aggregations |
| Dashboard | `/mnt/superstore/dashboard/` | PNG | Charts |
| Delta | `abfss://superstore@sachinxcube.dfs.core.windows.net/delta/` | Delta Lake | Queryable + time-travel |